# Part 9 - Model Diagnostics, SHAP & Calibration

In [ ]:
# !pip install shap -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.metrics import (
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    brier_score_loss
)
from sklearn.calibration import calibration_curve


## Load OOF Predictions

In [ ]:
# Expected from previous notebooks:
# oof_cat, oof_lgb, oof_xgb, y

oof = pd.DataFrame({
    "CatBoost": oof_cat,
    "LightGBM": oof_lgb,
    "XGBoost": oof_xgb,
    "Target": y
})

display(oof.head())


## Compare Individual Models

In [ ]:
for c in ["CatBoost","LightGBM","XGBoost"]:
    print(c, roc_auc_score(y,oof[c]))


## ROC Curves

In [ ]:
plt.figure(figsize=(7,6))

for c in ["CatBoost","LightGBM","XGBoost"]:
    RocCurveDisplay.from_predictions(y,oof[c],name=c,ax=plt.gca())

plt.title("ROC Curves")
plt.show()


## Precision-Recall Curves

In [ ]:
plt.figure(figsize=(7,6))

for c in ["CatBoost","LightGBM","XGBoost"]:
    PrecisionRecallDisplay.from_predictions(
        y,oof[c],name=c,ax=plt.gca()
    )

plt.title("Precision-Recall Curves")
plt.show()


## Calibration

In [ ]:
plt.figure(figsize=(6,6))

for c in ["CatBoost","LightGBM","XGBoost"]:
    prob_true, prob_pred = calibration_curve(y,oof[c],n_bins=10)
    plt.plot(prob_pred,prob_true,label=c)

plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("Calibration")
plt.show()

for c in ["CatBoost","LightGBM","XGBoost"]:
    print(c,"Brier:",brier_score_loss(y,oof[c]))


## SHAP (CatBoost Example)

In [ ]:
# model should be final trained CatBoost model
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X.iloc[:5000])

shap.summary_plot(
    shap_values,
    X.iloc[:5000],
    max_display=20
)


## Prediction Distribution

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(oof_cat,bins=50,alpha=.5,label='CatBoost')
plt.hist(oof_lgb,bins=50,alpha=.5,label='LightGBM')
plt.hist(oof_xgb,bins=50,alpha=.5,label='XGBoost')
plt.legend()
plt.title("Prediction Distribution")
plt.show()


## Error Analysis

In [ ]:
ensemble=(oof_cat+oof_lgb+oof_xgb)/3

errors=pd.DataFrame({
    "target":y,
    "prediction":ensemble
})

errors["abs_error"]=abs(errors.target-errors.prediction)

display(
    errors.sort_values("abs_error",ascending=False).head(20)
)


## Save OOF Predictions

In [ ]:
oof.to_csv("oof_predictions.csv",index=False)
print("Saved oof_predictions.csv")


## Final Checklist

- Compare individual model ROC-AUC
- Inspect SHAP importance
- Check calibration
- Analyze worst predictions
- Keep OOF predictions for stacking

The final notebook can use these diagnostics to refine features, tune models further, or build a stacked meta-model.
